# 04 — La carte : l'architecture Transformer, et ce que nous allons en garder

**Chapitre 2** · durée conseillée : 20 min · pas d'entraînement

![Parcours](figures/00_parcours.svg)

Avant de plonger dans les pièces (attention, FeedForward, résidus), regardons **le plan complet**. Ce notebook est volontairement court en code : son but est que chacun sache, à tout moment de la suite, **où se trouve la pièce que l'on construit**.

**À la fin de ce notebook, vous saurez :**
1. lire le schéma du Transformer original (encodeur + décodeur) ;
2. expliquer la différence entre **encodeur**, **décodeur** et **cross-attention** ;
3. dire pourquoi un GPT est un **decoder-only** ;
4. énumérer les différences entre le papier de 2017 et notre mini-GPT.

## 1. Le Transformer original (Vaswani et al., 2017)

Le Transformer a été conçu pour la **traduction automatique** : une phrase en entrée (anglais), une phrase en sortie (allemand). Il a deux moitiés :

<img src="https://upload.wikimedia.org/wikipedia/commons/3/34/Transformer%2C_full_architecture.png" alt="Architecture originale encodeur-décodeur d'un Transformer" width="560" style="background:white"/>

*Illustration : dvgodoy, [Wikimedia Commons — Transformer, full architecture](https://commons.wikimedia.org/wiki/File:Transformer,_full_architecture.png), [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/), reproduite sans modification.*

**Comment lire le schéma :**

- **L'encodeur** (à gauche) lit **toute la phrase source**. Chaque mot peut regarder **tous** les autres, avant comme après lui : l'attention est **bidirectionnelle**, sans masque. Il produit une représentation de la phrase.
- **Le décodeur** (à droite) écrit la traduction **mot après mot**. Il contient deux attentions :
  1. une **self-attention masquée** : chaque mot de la traduction ne peut regarder que les mots **déjà écrits** ;
  2. une **cross-attention** : les *queries* viennent du décodeur, les *keys* et *values* viennent de l'**encodeur**. C'est là que la traduction « consulte » la phrase source.
- Chaque bloc contient aussi un **FeedForward**, des **connexions résiduelles** (« Add ») et des **LayerNorm** (« Norm »), et le tout est répété `N = 6` fois.

## 2. Encodeur-décodeur vs décodeur seul

![Encodeur-décodeur vs GPT](figures/04_encodeur_decodeur_vs_gpt.svg)

Notre tâche n'est pas de traduire mais de **continuer un texte**. Il n'y a **pas de phrase source** à encoder : il n'y a que le texte déjà écrit. On supprime donc :

- tout l'**encodeur** ;
- la **cross-attention** du décodeur (elle n'aurait rien à consulter).

Il reste un empilement de blocs « self-attention masquée + FeedForward » : c'est un **Transformer decoder-only**, l'architecture des modèles GPT.

| Famille | Attention | Exemple | Tâche typique |
|---|---|---|---|
| **Encoder-only** | bidirectionnelle (pas de masque) | BERT | classer, comprendre un texte entier |
| **Encoder-decoder** | encodeur bidirectionnel + décodeur masqué + cross-attention | Transformer original, T5 | traduire, résumer |
| **Decoder-only** | masquée (causale) | GPT-2, GPT-3, LLaMA, **notre mini-GPT** | continuer un texte |

> **Pour aller plus loin :** dans sa vidéo, Karpathy consacre une partie à cette distinction (chapitre *« encoder vs. decoder vs. both (?) Transformers »*, vers la fin). Le support `slides_encodeur_decodeur.md` reprend ces points sous forme de diapositives.

**Une remarque qui servira au notebook 05 :** la seule différence de code entre une self-attention d'encodeur et une self-attention de décodeur est **une ligne** : celle qui applique le masque causal.

## 3. Ce que nous allons implémenter

![Le mini-GPT](figures/04_decoder_only.svg)

Lisez le schéma **de bas en haut**, en suivant les formes des tenseurs :

| Étape | Opération | Forme | Notebook |
|---|---|---|---|
| Entrée | indices des caractères | `(B, T)` | 01, 03 |
| Embeddings | token **+** position | `(B, T, C)` | 02, 08 |
| Bloc × `n_layer` | LayerNorm → **attention masquée multi-têtes** → `+` | `(B, T, C)` | 05, 07 |
|  | LayerNorm → **FeedForward** → `+` | `(B, T, C)` | 06, 07 |
| Sortie | LayerNorm finale → `lm_head` (Linear `C → V`) | `(B, T, V)` | 08 |
| Loss | reshape + `F.cross_entropy` | scalaire | 02, 03 |

**Ce qui est déjà acquis :** l'entrée, la sortie et la loss sont **identiques au bigramme**. Le Transformer ne change que ce qui se passe **au milieu** : la façon de produire les logits.

**L'idée directrice de Karpathy :** dans chaque bloc, on alterne
- **communication** (attention) : les positions échangent de l'information, mais seulement vers le futur → passé ;
- **calcul** (FeedForward) : chaque position traite seule ce qu'elle a reçu.

## 4. Où sont les paramètres ?

Sans entrer dans le code (ce sera le notebook 08), instancions le modèle de `minigpt.py` et regardons où se trouvent les paramètres entraînables.

In [1]:
# Préparation : imports et module partagé du chapitre
import math
import torch
import torch.nn as nn
from torch.nn import functional as F
import matplotlib.pyplot as plt

import minigpt  # fichier minigpt.py, à côté de ce notebook
torch.manual_seed(1337)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("PyTorch", torch.__version__, "| périphérique :", device)

V = 65            # vocabulaire de Shakespeare
modele = minigpt.GPTLanguageModel(vocab_size=V, block_size=64, n_embd=64, n_head=4, n_layer=4)

total = minigpt.compter_parametres(modele)
print(f"{'composant':38s} {'paramètres':>10s}   part")
lignes = [("embedding des tokens (V × C)", modele.token_embedding_table),
          ("embedding des positions (T × C)", modele.position_embedding_table)]
for i, bloc in enumerate(modele.blocks):
    lignes += [(f"bloc {i} : attention multi-têtes", bloc.sa),
               (f"bloc {i} : FeedForward", bloc.ffwd),
               (f"bloc {i} : 2 LayerNorm", nn.ModuleList([bloc.ln1, bloc.ln2]))]
lignes += [("LayerNorm finale", modele.ln_f), ("lm_head (C × V + V)", modele.lm_head)]
for nom, module in lignes:
    n = minigpt.compter_parametres(module)
    print(f"{nom:38s} {n:10,}   {100 * n / total:4.1f} %")
print(f"{'TOTAL':38s} {total:10,}")

PyTorch 2.14.1+cu130 | périphérique : cpu
composant                              paramètres   part
embedding des tokens (V × C)                4,160    2.0 %
embedding des positions (T × C)             4,096    1.9 %
bloc 0 : attention multi-têtes             16,448    7.8 %
bloc 0 : FeedForward                       33,088   15.6 %
bloc 0 : 2 LayerNorm                          256    0.1 %
bloc 1 : attention multi-têtes             16,448    7.8 %
bloc 1 : FeedForward                       33,088   15.6 %
bloc 1 : 2 LayerNorm                          256    0.1 %
bloc 2 : attention multi-têtes             16,448    7.8 %
bloc 2 : FeedForward                       33,088   15.6 %
bloc 2 : 2 LayerNorm                          256    0.1 %
bloc 3 : attention multi-têtes             16,448    7.8 %
bloc 3 : FeedForward                       33,088   15.6 %
bloc 3 : 2 LayerNorm                          256    0.1 %
LayerNorm finale                              128    0.1 %
lm_head (C × V +

**Observations :**
- La majorité des paramètres est dans les **blocs**, et dans chaque bloc le **FeedForward** pèse environ **deux fois** l'attention (il élargit à `4C`, notebook 06).
- Le masque causal n'apparaît pas : ce n'est **pas** un paramètre.
- ~200 000 paramètres : c'est minuscule (GPT-2 small : 124 M ; GPT-3 : 175 G), mais **l'architecture est la même**.

## 5. Notre mini-GPT n'est pas une copie exacte du papier

| | Transformer (2017) | Notre mini-GPT |
|---|---|---|
| Structure | encodeur + décodeur | **décodeur seul**, sans cross-attention |
| Tokens | sous-mots (BPE, ~37 000) | **caractères** (65 pour Shakespeare) |
| Position | sinusoïdes **fixes** | embedding de position **appris** (comme GPT-2) |
| Normalisation | *post-norm* : `LayerNorm(x + f(x))` | *pré-norm* : `x + f(LayerNorm(x))` (comme GPT-2) |
| Tailles | `d_model = 512`, `h = 8`, `N = 6`, `d_ff = 2048` | `C = 64`, `h = 4`, `n_layer = 4`, `4C = 256` |
| Activation | ReLU | ReLU (GPT-2 utilise GELU) |

Ces différences ne changent pas le **principe**. Le *pré-norm* est devenu la norme car il rend l'entraînement des réseaux profonds plus stable (Xiong et al., 2020) ; nous verrons pourquoi au notebook 07.

## À retenir

- Transformer original = **encodeur** (attention bidirectionnelle) + **décodeur** (attention masquée + cross-attention vers l'encodeur).
- **GPT = decoder-only** : pas de texte source → pas d'encodeur, pas de cross-attention.
- Un bloc = **communication** (attention masquée) puis **calcul** (FeedForward), avec résidus et LayerNorm, répété `n_layer` fois.
- Entrée, sortie et loss sont celles du bigramme : seul **le milieu** change.

**Questions de vérification**
1. Pourquoi la self-attention de l'encodeur n'a-t-elle pas besoin de masque ?
2. Dans la cross-attention, d'où viennent Q ? Et K, V ?
3. Quelles parties du schéma original disparaissent dans un GPT ?

**Suite → 05 :** la pièce centrale, l'**attention masquée**.

### Références
- Vaswani et al., [*Attention Is All You Need*](https://arxiv.org/abs/1706.03762), NeurIPS 2017 — figure 1 et §3.
- Andrej Karpathy, [*Let's build GPT: from scratch, in code, spelled out*](https://www.youtube.com/watch?v=kCc8FmEb1nY) (2023) et son code [ng-video-lecture](https://github.com/karpathy/ng-video-lecture).
- A. Radford et al., [*Improving Language Understanding by Generative Pre-Training*](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) (GPT-1, 2018) et [*Language Models are Unsupervised Multitask Learners*](https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf) (GPT-2, 2019).
- J. Devlin et al., [*BERT*](https://arxiv.org/abs/1810.04805), 2018 — l'exemple type d'encoder-only.
- R. Xiong et al., [*On Layer Normalization in the Transformer Architecture*](https://arxiv.org/abs/2002.04745), ICML 2020 — pré-norm vs post-norm.
- J. Alammar, [*The Illustrated Transformer*](https://jalammar.github.io/illustrated-transformer/) — excellente explication visuelle de l'encodeur-décodeur.
- A. Rush, [*The Annotated Transformer*](https://nlp.seas.harvard.edu/annotated-transformer/) — le papier ligne à ligne, avec le code.